# [Agent] NB2 — SFT / GRPO Training\nSection 10(ChatML 준비) + 11(LoRA SFT/GRPO) + 12(adapter GCS 업로드).\n검증 산출물(Act2): train/val loss, GRPO reward 4-axis + KL, output validity rate.\nGRPO로 바꾸려면 아래 override에서 USE_GRPO=True.

**v3 — EXPERIMENT_STAGE** (Bootstrap 셀 한 줄로 전환, NB01-04와 동일 stem):

| Stage | mode | TRAIN_MODE | SUF_BALANCE | GRPO | jsonl stem |
|---|---|---|---|---|---|
| S1 | forward | forward_full | original | ✗ | training_data_forward_s1 |
| S2 | forward | forward_full | original | ✗ | training_data_forward |
| S3 | agent | wm_focus(3key) | downsample (A/B: ambiguous_only) | ✗ | training_data_mined_3key |
| S4 | agent | wm_only(1key) | downsample | ✓ (agent) | training_data_mined |

**GRPO 확정 버그 수정(cell Section 11-B)**: reward 최상단 cls 검사를 forward 분기로 이동 —
`wm_only`(1-key)는 `{"what_missing": ...}`만 출력하므로 기존 코드에서 전 reward=0 → adv_std=0 → 전 스텝 skip.

## Section 0: 환경

In [20]:
# GCP 인증 — gcsfuse 마운트를 위해 여전히 필요
# (Groq API는 별도 API key로 동작, GCP 인증과 무관)
from google.colab import auth
auth.authenticate_user()
print('✅ GCP 인증 완료 (gcsfuse 마운트용)')

# 2. gcsfuse 설치
!echo "deb http://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!sudo apt-get update
!sudo apt-get install gcsfuse

# gcsfuse 설치 & 버킷 마운트
import os

GCS_BUCKET  = "fraud_ecomm"
GCP_PROJECT = "kakaoo-501107"
MOUNT_POINT = "/content/gcs_mount"

!apt-get install -y -q gcsfuse

os.makedirs(MOUNT_POINT, exist_ok=True)

# gcsfuse 마운트 (ADC 자격증명 자동 사용)
!gcsfuse --implicit-dirs \
    --key-file="" \
    {GCS_BUCKET} {MOUNT_POINT} 2>&1 | tail -3

# 마운트 확인
result = os.listdir(MOUNT_POINT)
print(f'✅ GCS 마운트 완료: {MOUNT_POINT}')
print(f'   최상위 폴더: {result[:5]}')

# !pip install -U huggingface_hub[hf_xet]

# v5: 공통 패키지 일괄 설치 (provider 무관)
!pip install -q \
    trl peft transformers accelerate datasets \
    groq google-genai \
    bitsandbytes

print("✅ 설치 완료")

✅ GCP 인증 완료 (gcsfuse 마운트용)
deb http://packages.cloud.google.com/apt gcsfuse-jammy main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: apt-key is deprecated. Manage keyring files in trusted.gpg.d instead (see apt-key(8)).
100  1022  100  1022    0     0  11924      0 --:--:-- --:--:-- --:--:-- 12023
OK
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://packages.cloud.google.com/apt gcsfuse-jammy InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubun

## Bootstrap: 공통 모듈 + 데이터 로드

In [21]:
# ============================================================
# 공통 모듈(common/*.py) 로드 — 버킷에서 로컬로 복사 후 import
# (편집-반영 안정 + import 캐시 회피). 버킷 경로는 환경에 맞게 1회 수정.
# ============================================================
import sys, os, shutil, importlib

COMMON_SRC = "/content/gcs_mount/tree-search/analyzer_ft/common"   # ← .py 업로드 위치
COMMON_DST = "/content/common"
shutil.rmtree(COMMON_DST, ignore_errors=True)
shutil.copytree(COMMON_SRC, COMMON_DST)
if COMMON_DST not in sys.path:
    sys.path.insert(0, COMMON_DST)

import config, prompts, data_io, schema
for _m in (config, prompts, data_io, schema):
    importlib.reload(_m)
print("✅ common 모듈 로드:", [m.__name__ for m in (config, prompts, data_io, schema)])


✅ common 모듈 로드: ['config', 'prompts', 'data_io', 'schema']


In [22]:
# ============================================================
# 데이터 로드 + 런타임 바인딩 (mode="agent")
# config 상수를 펼치고, data_io.load_all()이 데이터/맵을 주입.
# ============================================================
from config import *
PHASE = "train"
# ════════ EXPERIMENT_STAGE — NB01-04(v3)와 동일한 4단계 스위치 ════════
EXPERIMENT_STAGE = "S3"     # ★ "S1" | "S2" | "S3" | "S4"
_STAGE_CFG = {
    "S1": dict(mode="forward", train_mode="forward_full", use_grpo=False,
               suf_balance="original",   stem="training_data_forward_s1"),
    "S2": dict(mode="forward", train_mode="forward_full", use_grpo=False,
               suf_balance="original",   stem="training_data_forward"),
    "S3": dict(mode="agent",   train_mode="wm_focus",     use_grpo=False,
               suf_balance="downsample", stem="training_data_mined_3key"),   # A/B: "ambiguous_only"
    "S4": dict(mode="agent",   train_mode="wm_only",      use_grpo=True,
               suf_balance="downsample", stem="training_data_mined"),
}[EXPERIMENT_STAGE]
mode       = _STAGE_CFG["mode"]
STAGE_STEM = _STAGE_CFG["stem"]
# 어댑터 저장 태그: S2/S4는 기존 규칙(mode_) 유지, S1/S3은 stage 접미로 충돌 방지
model_tag  = f"{mode}_" if EXPERIMENT_STAGE in ("S2", "S4") else f"{mode}_{EXPERIMENT_STAGE.lower()}_"
print(f"■ EXPERIMENT_STAGE={EXPERIMENT_STAGE}  {_STAGE_CFG}")
globals().update(config.set_data(mode=mode))

# ── ★ STAGE stem으로 학습/평가 jsonl 유일 경로 확정 (stale 후보 탐색 버그 재발 방지) ──
import os as _os
_base_dir = _os.path.dirname(TRAINING_JSONL_PATH)
TRAINING_JSONL_PATH = _os.path.join(_base_dir, f"{STAGE_STEM}.jsonl")
EVAL_JSONL_PATH     = _os.path.join(_base_dir, f"{STAGE_STEM}_eval.jsonl")
print(f"■ train: {TRAINING_JSONL_PATH}\n■ eval : {EVAL_JSONL_PATH}")

config.ensure_dirs()
config.print_config()

globals().update(data_io.load_all(mode=mode))   # mode="agent" or "forward"
from prompts import (SYSTEM_PROMPT, build_sample, extract_test_features,
                     build_analyzer_user_prompt, build_teacher_prompt_with_gt,
                     compute_l2_distances, format_top5_neighbors,
                     format_transaction, format_persona, normalize_dist, dist_tag)
print("✅ 런타임 준비 완료 (build_sample / build_analyzer_user_prompt 사용 가능)")

# ============================================================
# LLM 설정
# ============================================================

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# ★ stage가 결정 (필요 시 아래 줄로 수동 override)
TRAIN_MODE = _STAGE_CFG["train_mode"]   # "cls" | "wm_focus" | "wm_only" | "forward_full"
# True: sufficiency를 입력에 deterministic 주입 + what_missing 집중
WM_FOCUS    = TRAIN_MODE in ("wm_focus", "wm_only")
# 하위호환: CLS_ONLY 참조하는 코드용
CLS_ONLY = (TRAIN_MODE == "cls")
# ★ v2: 모드별 max_new — evaluator wm(반구조화 "<direction>: <evidence>")≈40,
#   forward 3-key(reasoning ≤25 words)≈56. decode가 대역폭 바운드라 직결 절감.
GEN_MAX_NEW = {"cls": 32, "wm_focus": 40, "wm_only": 32, "forward_full": 56}.get(TRAIN_MODE, 64)

_KEEP_KEYS = {"cls": ["cls"],
              "cls_suf": ["cls", "sufficiency"],
              "wm_only":      ["what_missing"],                      # ★ 1-key
              "wm_focus": ["cls", "sufficiency", "what_missing"],
              "full": ["cls", "sufficiency", "next_action", "what_missing"],
              "forward_full": ["cls", "confidence", "reasoning"]}[TRAIN_MODE]

# ── [삽입] sufficiency 균형 조정: null collapse 방지 ──
SUF_BALANCE     = _STAGE_CFG["suf_balance"]   # "original" | "downsample" | "ambiguous_only"
#   S3 A/B 재현: SUF_BALANCE="ambiguous_only" (nd_clear 53.8% 붕괴) vs "downsample"(94.7%)
CONFIDENT_RATIO = 1.0            # downsample 전용: ambiguous 대비 confident 배수

# ── GRPO 설정 (task 공용 / GPU 제약 → 작게 시작) ──
USE_GRPO = _STAGE_CFG["use_grpo"]  # stage가 결정 (S2 forward GRPO 실험 시 수동 True)
GRPO_FROM_SAVED_SFT = False   # ★ True: GCS의 SFT adapter를 시작점으로 GRPO만 실행 (SFT 재학습 생략)


GRPO_TASK            = "forward" if mode == "forward" else "agent"
GRPO_NUM_GENERATIONS = 6        # G: VRAM 여유 확인 후 4→8 단계적 증가
GRPO_BATCH_SIZE      = 1        # prompt B (실제 시퀀스 = B×G)
GRPO_EPOCHS          = 1
GRPO_MAX_STEPS       = None       # smoke용 step 상한: 20 (None이면 전량)
GRPO_LR              = 5e-6
GRPO_KL_COEF         = 0.05
GRPO_TEMPERATURE     = 1.1      # defualt : 1.1
GRPO_MAX_NEW         = 80              # agent 40 / forward 56
GRPO_LEN_BUDGET      = 56
GRPO_AMBIGUOUS_ONLY  = (GRPO_TASK == "agent")   # confident는 자유출력 없음(wm=null)

# 첫 실행은 USE_GRPO=True, G=2, B=1, MAX_STEPS=20으로 smoke → VRAM peak과 step 시간 확인 후 G를 4로 올리시면 됩니다.

■ EXPERIMENT_STAGE=S3  {'mode': 'agent', 'train_mode': 'wm_focus', 'use_grpo': False, 'suf_balance': 'downsample', 'stem': 'training_data_mined_3key'}
■ train: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_mined_3key.jsonl
■ eval : /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_mined_3key_eval.jsonl
PHASE            : generate
EXPERIMENT_CASE  : pca_64_k100
TEACHER_PROVIDER : huggingface
TEACHER_MODEL    : Qwen/Qwen2.5-14B-Instruct
GENERATION_MODE  : mining_only
TRAIN_ON         : mining_only
TRAINING_JSONL   : /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_mined.jsonl
OUT_ADAPTER_PATH : /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/adapter
✅ data_io.load_all(mode=agent)
   personas=100  coreset_map=122,810
   test=28,302  normal/abnormal=14151/14151
   dist p25/p50/p75=1.264/1.448/1.651  IQR=0.387
✅ 런타임 준비 완료 (build_sample / build_analyzer_user_prompt 사용 가능)


## Section 10: SFT 데이터셋 준비 (ChatML)

In [23]:
# ============================================================
# Section 10: ChatML 포맷 변환 및 Train/Eval Dataset 로드
# ============================================================

from datasets import Dataset
import json as _json
# from typing import List

def _load_jsonl(path: str) -> List[Dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip(): records.append(json.loads(line))
    print(f"  📂 로드: {path}  ({len(records)}개)")
    return records

def _suf_from_nd(nd):
    return "ambiguous" if (0.3 <= nd < 0.55) else "confident"

def to_chatml(record):
    user = record['user']
    asst = record['assistant']
    if WM_FOCUS:
        nd = record['metadata'].get('top1_l2_dist_norm', -1)
        suf = _suf_from_nd(nd) if nd >= 0 else "confident"
        # ① 입력에 deterministic sufficiency 주입 (student는 추론 안 함)
        # user = user + f"\n\n[SYSTEM-COMPUTED] retrieval_sufficiency = {suf.upper()}"
        if "retrieval_sufficiency =" not in user:
            user = user + f"\n\n[SYSTEM-COMPUTED] retrieval_sufficiency = {suf.upper()}"
        # ② 출력은 what_missing 집중: cls + what_missing만 (sufficiency 제거 — 입력에 줬으니)
        try:
            d = _json.loads(asst)
            # d = {"cls": d["cls"], "what_missing": d.get("what_missing")}
            d = {k: d.get(k) for k in _KEEP_KEYS}
            asst = _json.dumps(d, ensure_ascii=False)
        except Exception:
            pass
    text = (f"<|im_start|>system\n{record['system']}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{asst}<|im_end|>")
    return {"text": text}
# ── 런타임 재시작 후 학습 전용 실행 경로 ─────────────
print(f"[PHASE=train] jsonl 로드: {TRAINING_JSONL_PATH}")
training_records = _load_jsonl(TRAINING_JSONL_PATH)

if SUF_BALANCE != "original":
    import random as _r
    _sufof = lambda r: (json.loads(r["assistant"]).get("sufficiency")
                        or _suf_from_nd(r["metadata"].get("top1_l2_dist_norm", -1)))
    _amb = [r for r in training_records if _sufof(r) == "ambiguous"]
    _con = [r for r in training_records if _sufof(r) == "confident"]
    if SUF_BALANCE == "ambiguous_only":
        _keep = []
    else:
        _n_keep, _by_cls = round(len(_amb) * CONFIDENT_RATIO), {}
        for r in _con:                                   # cls 비율 보존 층화 샘플링
            _by_cls.setdefault(json.loads(r["assistant"])["cls"], []).append(r)
        _keep = []
        for _c, _recs in _by_cls.items():
            _r.Random(42).shuffle(_recs)
            _keep += _recs[:max(1, round(_n_keep * len(_recs) / max(len(_con), 1)))]
    training_records = _amb + _keep
    _r.Random(42).shuffle(training_records)
    _n_null = sum(1 for r in training_records if not json.loads(r["assistant"]).get("what_missing"))
    print(f"[suf balance:{SUF_BALANCE}] ambiguous={len(_amb)} + confident={len(_keep)} "
          f"→ {len(training_records)}개  (null 비율 {_n_null/max(len(training_records),1):.1%})")

# ── 학습 데이터 통계 출력 ──────────────────────────────────────────
cls_list  = [json.loads(r['assistant']).get('cls', '?') for r in training_records]   # ✅ cls
cat_list  = [r['metadata']['category']         for r in training_records]
top1_list = [r['metadata']['top1_l2_distance'] for r in training_records]

print(cls_list)
print(cat_list)
print(top1_list)

print(json.dumps(training_records[0], indent=2, ensure_ascii=False)[:600] + "...")

# ★ GRPO용 추가 메타데이터 (gt_label은 이미 metadata에 있음 → 그대로 사용)
for record in training_records:
    # ✅ gt_label: assistant 파싱이 아니라 metadata의 GT 사용 (rejection sampling이 일치 보장)
    if 'gt_label' not in record['metadata']:
        # 안전망: 혹시 없으면 cls에서 유도
        cls = json.loads(record['assistant']).get('cls', 'NORMAL')
        record['metadata']['gt_label'] = 1 if cls == 'ABNORMAL' else 0
    # user_prompt 추출 (GRPO용)
    record['metadata']['user_prompt'] = record['user']

print(f"\n✅ GRPO 메타데이터 확인 완료")
print(f"   gt_label 샘플: {[r['metadata']['gt_label'] for r in training_records[:5]]}")

# ──────────────────────────────────────────────────────────
# Train Dataset: training_records 사용 (Section 9에서 생성)
# ──────────────────────────────────────────────────────────
train_recs = training_records
train_dataset = Dataset.from_list([to_chatml(r) for r in train_recs])

print(f"✅ Train dataset: {len(train_dataset):,} samples")


# ──────────────────────────────────────────────────────────
# Eval Dataset: 별도 JSONL 로드
# ──────────────────────────────────────────────────────────
import os
# ── ★ stale-후보 버그 수정: agent/forward 후보 우선순위 탐색 제거 →
#    STAGE_STEM에서 유도된 EVAL_JSONL_PATH 단일 경로 (다른 stage 파일 오적재 원천 차단) ──
eval_jsonl_path = globals().get("EVAL_JSONL_PATH") or TRAINING_JSONL_PATH.replace(
    ".jsonl", "_eval.jsonl")
if not os.path.exists(eval_jsonl_path):
    raise FileNotFoundError(
        f"eval jsonl 없음: {eval_jsonl_path}\n"
        f"→ NB01-04(v3)를 동일 EXPERIMENT_STAGE={globals().get('EXPERIMENT_STAGE','?')}로 실행해 생성"
        " (eval은 train split이 아니라 별도 추론 패스가 원칙)")
print(f"✅ eval 사용[{globals().get('EXPERIMENT_STAGE','?')}]: {eval_jsonl_path}")


if os.path.exists(eval_jsonl_path):
    print(f"\n✅ Loading eval dataset from: {eval_jsonl_path}")

    eval_recs = []
    with open(eval_jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            eval_recs.append(json.loads(line))

    eval_dataset = Dataset.from_list([to_chatml(r) for r in eval_recs])

    print(f"✅ Eval dataset: {len(eval_dataset):,} samples")
    print(f"   NORMAL:   {sum(1 for r in eval_recs if r['metadata']['gt_label']==0):,}")
    print(f"   ABNORMAL: {sum(1 for r in eval_recs if r['metadata']['gt_label']==1):,}")

else:
    # (도달 불가 — 위에서 raise. eval을 train split으로 만드는 것은 금지: 별도 추론 패스 원칙)
    print(f"\n⚠️  Eval JSONL not found: {eval_jsonl_path}")
    print(f"   Falling back to 10% split from training data")

    import random as _rnd
    _rnd.seed(42)
    shuffled = training_records[:]
    _rnd.shuffle(shuffled)

    n_eval = max(1, int(len(shuffled) * 0.10))
    eval_recs = shuffled[:n_eval]
    train_recs = shuffled[n_eval:]

    train_dataset = Dataset.from_list([to_chatml(r) for r in train_recs])
    eval_dataset = Dataset.from_list([to_chatml(r) for r in eval_recs])

    print(f"   Train: {len(train_dataset):,}  |  Eval: {len(eval_dataset):,}")


# ──────────────────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Final Dataset Sizes:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Eval:  {len(eval_dataset):,}")
print(f"{'='*60}")

print("\n[ChatML 샘플 (처음 500자)]")
print(train_dataset[0]['text'][:500])

[PHASE=train] jsonl 로드: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_mined_3key.jsonl
  📂 로드: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_mined_3key.jsonl  (1556개)
[suf balance:downsample] ambiguous=498 + confident=498 → 996개  (null 비율 50.0%)
['NORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'NORMAL', 'NORMAL', 'ABNORMAL', 'ABNORMAL', 'ABNORM

In [24]:
# ============================================================
# Training Data 통계 분석 (GT Label × Distance 관점)
# ============================================================

import json
import numpy as np
from collections import defaultdict

# ──────────────────────────────────────────────────────
# 1. JSONL 로드 & 파싱
# ──────────────────────────────────────────────────────
def load_and_analyze_jsonl(filepath, dataset_name):
    """JSONL 파일 통계 분석"""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))

    print(f"\n{'='*70}")
    print(f"{dataset_name} 데이터 분석 ({len(data):,}개)")
    print(f"{'='*70}")

    # ──────────────────────────────────────────────────────
    # 2. GT Label 분포
    # ──────────────────────────────────────────────────────
    gt_labels = [rec['metadata']['gt_label'] for rec in data]
    n_normal = gt_labels.count(0)
    n_abnormal = gt_labels.count(1)

    print(f"\n📌 GT Label 분포:")
    print(f"   NORMAL (label=0):   {n_normal:4,}개 ({n_normal/len(data)*100:5.1f}%)")
    print(f"   ABNORMAL (label=1): {n_abnormal:4,}개 ({n_abnormal/len(data)*100:5.1f}%)")

    # ──────────────────────────────────────────────────────
    # 3. Category 분포
    # ──────────────────────────────────────────────────────
    categories = defaultdict(int)
    for rec in data:
        cat = rec['metadata'].get('category', 'unknown')
        categories[cat] += 1

    print(f"\n📌 Category 분포:")
    for cat, count in sorted(categories.items()):
        print(f"   {cat:25s}: {count:4,}개 ({count/len(data)*100:5.1f}%)")

    # ──────────────────────────────────────────────────────
    # 4. Distance 분포
    # ──────────────────────────────────────────────────────
    norm_dists = [rec['metadata']['top1_l2_dist_norm'] for rec in data]

    dist_ranges = {
        '< 0.3 (Very similar)': sum(1 for d in norm_dists if d < 0.3),
        '0.3-0.5 (Moderate)': sum(1 for d in norm_dists if 0.3 <= d < 0.5),
        '>= 0.5 (Weak)': sum(1 for d in norm_dists if d >= 0.5),
    }

    print(f"\n📌 Distance 분포:")
    for range_name, count in dist_ranges.items():
        print(f"   {range_name:25s}: {count:4,}개 ({count/len(data)*100:5.1f}%)")

    print(f"\n   Distance 통계:")
    print(f"   평균: {np.mean(norm_dists):.3f}")
    print(f"   중앙: {np.median(norm_dists):.3f}")
    print(f"   범위: {np.min(norm_dists):.3f} - {np.max(norm_dists):.3f}")

    # ──────────────────────────────────────────────────────
    # 5. ✅ 교차 분석 (GT Label × Distance Range)
    # ──────────────────────────────────────────────────────
    print(f"\n📌 GT Label × Distance 교차 분석:")
    print(f"   {'Distance Range':20s} | {'NORMAL (0)':12s} | {'ABNORMAL (1)':12s} | Total")
    print(f"   {'-'*20}-+-{'-'*12}-+-{'-'*12}-+-{'-'*6}")

    for range_name in ['< 0.3', '0.3-0.5', '>= 0.5']:
        if range_name == '< 0.3':
            mask = [d < 0.3 for d in norm_dists]
        elif range_name == '0.3-0.5':
            mask = [0.3 <= d < 0.5 for d in norm_dists]
        else:
            mask = [d >= 0.5 for d in norm_dists]

        normal_count = sum(1 for i, m in enumerate(mask) if m and gt_labels[i] == 0)
        abnormal_count = sum(1 for i, m in enumerate(mask) if m and gt_labels[i] == 1)
        total_count = normal_count + abnormal_count

        print(f"   {range_name:20s} | {normal_count:4,} ({normal_count/total_count*100:5.1f}%) "
              f"| {abnormal_count:4,} ({abnormal_count/total_count*100:5.1f}%) "
              f"| {total_count:4,}")

    # ──────────────────────────────────────────────────────
    # 6. ✅ Category별 GT Label 분포
    # ──────────────────────────────────────────────────────
    if len(categories) > 1:
        print(f"\n📌 Category별 GT Label 분포:")
        print(f"   {'Category':25s} | {'NORMAL (0)':12s} | {'ABNORMAL (1)':12s} | Total")
        print(f"   {'-'*25}-+-{'-'*12}-+-{'-'*12}-+-{'-'*6}")

        for cat in sorted(categories.keys()):
            cat_data = [rec for rec in data if rec['metadata'].get('category') == cat]
            cat_labels = [rec['metadata']['gt_label'] for rec in cat_data]
            cat_normal = cat_labels.count(0)
            cat_abnormal = cat_labels.count(1)
            cat_total = len(cat_data)

            print(f"   {cat:25s} | {cat_normal:4,} ({cat_normal/cat_total*100:5.1f}%) "
                  f"| {cat_abnormal:4,} ({cat_abnormal/cat_total*100:5.1f}%) "
                  f"| {cat_total:4,}")

    return data

# ──────────────────────────────────────────────────────
# 7. 실행 (런타임 주입 변수 기반 동적 바인딩)
# ──────────────────────────────────────────────────────
import os

# config.py와 set_data()에서 주입된 전역 변수 및 파일명 활용
base_path = OUT_BASE
mined_filename = "training_data_mined.jsonl" if mode == "agent" else "training_data_forward.jsonl"

print(f"🔍 Current Mode: {mode.upper()} | Base Path: {base_path}")

# 1) Distance 데이터 로드 (파일이 존재할 때만)
_dist_path = f"{base_path}/training_data_distance.jsonl"
distance_data = load_and_analyze_jsonl(_dist_path, f"Distance 데이터 (Easy Samples) [{mode}]") if os.path.exists(_dist_path) else None

# 2) Mode별 Mined 데이터 로드
_mined_path = f"{base_path}/{mined_filename}"
mined_data = load_and_analyze_jsonl(_mined_path, f"Mined 데이터 (Hard Negative Mining) [{mode}]") if os.path.exists(_mined_path) else None

# 3) Eval 데이터 로드
_eval_path = f"{base_path}/training_data_eval.jsonl"
eval_data = load_and_analyze_jsonl(_eval_path, f"Eval 데이터 [{mode}]") if os.path.exists(_eval_path) else None


# ──────────────────────────────────────────────────────
# 8. 전체 요약 (데이터 유무에 따른 동적 예외 처리)
# ──────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"전체 요약 ({mode.upper()} 모드 결과)")
print(f"{'='*70}")

# 존재하는 데이터셋만 유연하게 합산
datasets_to_combine = []
if distance_data is not None: datasets_to_combine.append(distance_data)
if mined_data is not None:    datasets_to_combine.append(mined_data)
if eval_data is not None:     datasets_to_combine.append(eval_data)

if not datasets_to_combine:
    print("❌ 로드된 데이터셋이 없어 요약을 진행할 수 없습니다.")
    all_data = []
else:
    all_data = sum(datasets_to_combine, [])

all_labels = [rec['metadata']['gt_label'] for rec in all_data]
all_dists = [rec['metadata']['top1_l2_dist_norm'] for rec in all_data]

print(f"\n총 샘플 수: {len(all_data):,}개")
print(f"\nGT Label 분포:")
print(f"   NORMAL (0):   {all_labels.count(0):,}개 ({all_labels.count(0)/len(all_data)*100:.1f}%)")
print(f"   ABNORMAL (1): {all_labels.count(1):,}개 ({all_labels.count(1)/len(all_data)*100:.1f}%)")

print(f"\nDistance 분포:")
print(f"   < 0.3:   {sum(1 for d in all_dists if d < 0.3):,}개")
print(f"   0.3-0.5: {sum(1 for d in all_dists if 0.3 <= d < 0.5):,}개")
print(f"   >= 0.5:  {sum(1 for d in all_dists if d >= 0.5):,}개")

# ✅ Reward 설계를 위한 핵심 통계
print(f"\n{'='*70}")
print("⚠️ Reward 설계 핵심 통계")
print(f"{'='*70}")

print(f"\n1️⃣ NORMAL 샘플 (GT=0)의 Distance 분포:")
normal_dists = [all_dists[i] for i in range(len(all_data)) if all_labels[i] == 0]
print(f"   < 0.3:   {sum(1 for d in normal_dists if d < 0.3):,}개 ({sum(1 for d in normal_dists if d < 0.3)/len(normal_dists)*100:.1f}%)")
print(f"   0.3-0.5: {sum(1 for d in normal_dists if 0.3 <= d < 0.5):,}개 ({sum(1 for d in normal_dists if 0.3 <= d < 0.5)/len(normal_dists)*100:.1f}%)")
print(f"   >= 0.5:  {sum(1 for d in normal_dists if d >= 0.5):,}개 ({sum(1 for d in normal_dists if d >= 0.5)/len(normal_dists)*100:.1f}%)")

print(f"\n2️⃣ ABNORMAL 샘플 (GT=1)의 Distance 분포:")
abnormal_dists = [all_dists[i] for i in range(len(all_data)) if all_labels[i] == 1]
print(f"   < 0.3:   {sum(1 for d in abnormal_dists if d < 0.3):,}개 ({sum(1 for d in abnormal_dists if d < 0.3)/len(abnormal_dists)*100:.1f}%)")
print(f"   0.3-0.5: {sum(1 for d in abnormal_dists if 0.3 <= d < 0.5):,}개 ({sum(1 for d in abnormal_dists if 0.3 <= d < 0.5)/len(abnormal_dists)*100:.1f}%)")
print(f"   >= 0.5:  {sum(1 for d in abnormal_dists if d >= 0.5):,}개 ({sum(1 for d in abnormal_dists if d >= 0.5)/len(abnormal_dists)*100:.1f}%)")

print(f"\n3️⃣ Distance Range별 ABNORMAL 비율:")
for range_name, (low, high) in [("< 0.3", (0, 0.3)), ("0.3-0.5", (0.3, 0.5)), (">= 0.5", (0.5, 1.0))]:
    if high == 1.0:
        mask = [d >= low for d in all_dists]
    else:
        mask = [low <= d < high for d in all_dists]

    range_labels = [all_labels[i] for i, m in enumerate(mask) if m]
    if range_labels:
        abnormal_ratio = range_labels.count(1) / len(range_labels)
        print(f"   {range_name:8s}: {abnormal_ratio*100:5.1f}% ABNORMAL ({range_labels.count(1)}/{len(range_labels)})")

print(f"\n{'='*70}")

🔍 Current Mode: AGENT | Base Path: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713

Mined 데이터 (Hard Negative Mining) [agent] 데이터 분석 (1,844개)

📌 GT Label 분포:
   NORMAL (label=0):   1,026개 ( 55.6%)
   ABNORMAL (label=1):  818개 ( 44.4%)

📌 Category 분포:
   boundary_distance        : 1,066개 ( 57.8%)
   hard_negative_type1      :  394개 ( 21.4%)
   hard_negative_type2      :  384개 ( 20.8%)

📌 Distance 분포:
   < 0.3 (Very similar)     :  362개 ( 19.6%)
   0.3-0.5 (Moderate)       :  634개 ( 34.4%)
   >= 0.5 (Weak)            :  848개 ( 46.0%)

   Distance 통계:
   평균: 0.485
   중앙: 0.478
   범위: 0.080 - 0.955

📌 GT Label × Distance 교차 분석:
   Distance Range       | NORMAL (0)   | ABNORMAL (1) | Total
   ---------------------+--------------+--------------+-------
   < 0.3                |  361 ( 99.7%) |    1 (  0.3%) |  362
   0.3-0.5              |  535 ( 84.4%) |   99 ( 15.6%) |  634
   >= 0.5               |  130 ( 15.3%) |  718 ( 84.7%) |  848

📌 Category별 GT Label 분포:
   Category   

## Section 11: Qwen2.5-0.5B SFT/GRPO + LoRA

In [25]:
# ============================================================
# Teacher GPU 해제 — SFT 학습 전 실행
# ============================================================
# ※ HF teacher 모델이 GPU를 점유하고 있으면 student 학습 시 OOM 발생
#    Groq/Gemini는 클라이언트만 제거하면 됨

import gc, torch

def release_teacher_gpu(verbose: bool = True):
    global groq_client, gemini_client, hf_model, hf_tokenizer, hf_pipeline, _hf_current_model
    before = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

    # API 클라이언트 해제
    groq_client = gemini_client = None

    # HF 모델 해제
    for var_name in ["hf_model", "hf_tokenizer", "hf_pipeline"]:
        obj = globals().get(var_name)
        if obj is not None:
            del obj
    hf_model = hf_tokenizer = hf_pipeline = None
    _hf_current_model = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    after = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    if verbose:
        print(f"✅ Teacher GPU 해제 완료")
        print(f"   Before: {before:.2f} GB  →  After: {after:.2f} GB  (해제: {before-after:.2f} GB)")

release_teacher_gpu()

# ── GPU 상태 확인 ──────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated()  / 1024**3
    reserv = torch.cuda.memory_reserved()   / 1024**3
    total  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Allocated : {alloc:.2f} GB")
    print(f"  Reserved  : {reserv:.2f} GB")
    print(f"  Total     : {total:.1f} GB")
    print(f"  Available : {total - reserv:.2f} GB")
else:
    print("GPU 없음")

✅ Teacher GPU 해제 완료
   Before: 0.96 GB  →  After: 0.96 GB  (해제: 0.00 GB)
GPU: NVIDIA L4
  Allocated : 0.96 GB
  Reserved  : 1.00 GB
  Total     : 22.0 GB
  Available : 21.03 GB


In [26]:
# ============================================================
# Section 11: Qwen2.5-0.5B (Student) 로드
# ============================================================
# ── 재실행 안전 장치: 기존 student 모델 완전 제거 ──────────────────
import gc, torch
for _var in ["student_model", "student_tokenizer", "model", "tokenizer", "trainer"]:
    if _var in dir():
        exec(f"del {_var}")
gc.collect()
torch.cuda.empty_cache()
print(f"GPU 초기화 후: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ── 아래는 기존 코드 그대로 ────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

os.makedirs(LOCAL_ADAPTER, exist_ok=True)

print(f"[1/3] Student 모델 로드: {BASE_MODEL_ID}")
student_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
student_tokenizer.pad_token       = student_tokenizer.eos_token
student_tokenizer.truncation_side = "left"

# global student_model
student_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"  device: {next(student_model.parameters()).device}")
print(f"  params: {sum(p.numel() for p in student_model.parameters())/1e6:.1f}M")
if torch.cuda.is_available():
    print(f"  GPU 사용: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# 하위 호환용 alias
model = student_model
tokenizer = student_tokenizer

GPU 초기화 후: 0.02 GB
[1/3] Student 모델 로드: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  device: cuda:0
  params: 494.0M
  GPU 사용: 0.94 GB


In [27]:
# ── LoRA + SFT 설정 ──────────────────────────────────────────────
# GPU 가이드:
#   T4(16GB):  max_length=1024, batch=2, grad_accum=4, grad_ckpt=True
#   A100(40GB): max_length=1536, batch=4, grad_accum=2, grad_ckpt=False

# ── torchao 0.10.0 (peft 요구 0.16.0+) 버전 충돌 우회 ──
# peft가 torchao를 "미설치"로 인식하게 해 버전 체크를 건너뜀.
import importlib.metadata as _md
if not getattr(_md, "_torchao_patched", False):
    _orig_v = _md.version
    def _patched_version(n):
        return "0.16.0" if n == "torchao" else _orig_v(n)
    _md.version = _patched_version
    _md._torchao_patched = True
    print("✅ torchao version patch 적용")
else:
    print("⏭️ 이미 패치됨 (재적용 안 함)")


_has_gpu   = torch.cuda.is_available()
_total_gb  = torch.cuda.get_device_properties(0).total_memory/1024**3 if _has_gpu else 0
_is_large  = _total_gb >= 35   # A100/H100 계열

peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir=LOCAL_ADAPTER,
    dataset_text_field="text",
    max_length           = 1024,                # ✅ 3키 출력 짧음 → 768 충분 (속도↑)
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs     = 3,                  # 120개 → 5
    learning_rate        = 1e-4,
    lr_scheduler_type    = "cosine",
    warmup_steps         = 15,
    optim                = "adamw_torch",
    gradient_checkpointing = False,
    save_strategy        = "epoch",
    eval_strategy        = "epoch",
    logging_steps        = 5,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better    = False,
    report_to            = "none",
    bf16=True, fp16=False,
    packing=False,
    completion_only_loss = True,               # ✅ ★assistant만 loss (collapse 방지 핵심)
)

print(f"[2/3] SFT 설정 ({torch.cuda.get_device_name(0) if _has_gpu else 'CPU'})")
print(f"  GPU total        : {_total_gb:.1f} GB  ({'Large' if _is_large else 'Small'} 모드)")
print(f"  max_length       : {sft_config.max_length}")
print(f"  batch size       : {sft_config.per_device_train_batch_size}  (eff={sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps})")
print(f"  gradient_ckpt    : {sft_config.gradient_checkpointing}")
print(f"  optim            : {sft_config.optim}")

trainer = SFTTrainer(
    model=student_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    peft_config=peft_config,
)
trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
print(f"  Trainable params : {trainable/1e6:.2f}M")

⏭️ 이미 패치됨 (재적용 안 함)
[2/3] SFT 설정 (NVIDIA L4)
  GPU total        : 22.0 GB  (Small 모드)
  max_length       : 1024
  batch size       : 2  (eff=8)
  gradient_ckpt    : False
  optim            : OptimizerNames.ADAMW_TORCH


Adding EOS to train dataset:   0%|          | 0/996 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/996 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/996 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/996 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

  Trainable params : 2.16M


In [28]:
# ============================================================
# Section 11-B: GRPO Training (v4 — agent/forward 공용)
#   · reward = cls / grounded / direction(agent) | calib(forward) / format
#              − template · length · distance-restate 페널티  (전부 코드 검증 가능)
#   · reference = LoRA disable_adapter (deepcopy 제거 → VRAM 절반)
#   · advantage = 그룹(G) 내 정규화 = Group *Relative*
#   · agent는 ambiguous만 학습 (confident는 cls=nd 주입 / wm=null → 자유출력 없음)
#   · v3: 카테고리 평탄화(shuffle) + zero-variance skip + 첫 스텝 진단
#   · v4: ① cluster_mismatch에 z/value 인용 필수(reward hacking 차단)
#         ② neighbor_insufficient를 실제 differs 근거로 조임 (0/0 무료점수 제거)
#         ③ RETRIEVAL 라인 진단 출력
# ============================================================

def run_grpo_training():
    import torch
    import torch.nn.functional as F
    from torch.optim import AdamW
    from tqdm import tqdm
    import random as _r
    import re, time

    global student_model

    print("\n" + "=" * 60)
    print(f"GRPO: Group Relative Policy Optimization  [task={GRPO_TASK}]")
    print("=" * 60)

    # ── [0/5] Gradient checkpointing off (generate 시 cache 사용) ──
    print("\n[0/5] Disabling gradient checkpointing...")
    student_model.config.use_cache = True
    if hasattr(student_model, "gradient_checkpointing_disable"):
        student_model.gradient_checkpointing_disable()

    # ── [1/5] PEFT 핸들 확보 (TRL은 trainer.model에만 LoRA를 래핑) ──
    print("\n[1/5] Resolving PEFT model handle...")
    if not hasattr(student_model, "disable_adapter"):
        _cand = globals().get("trainer", None)
        _peft = getattr(_cand, "model", None) if _cand is not None else None
        assert _peft is not None and hasattr(_peft, "disable_adapter"), (
            "PEFT 모델을 찾을 수 없음 — SFT(cell 12/14)를 먼저 실행했는지 확인")
        student_model = _peft            # ★ 이후 전 구간이 PeftModel을 사용
    print(f"    ✅ PEFT handle: {type(student_model).__name__}")

    # ── [2/5] 그룹 구성 ──
    print("\n[2/5] Grouping samples...")
    if GRPO_AMBIGUOUS_ONLY:
        pool = [r for r in training_records
                if json.loads(r["assistant"]).get("sufficiency") == "ambiguous"]
    else:
        pool = list(training_records)
    grpo_groups = {}
    for r in pool:
        grpo_groups.setdefault(r["metadata"].get("category", "all"), []).append(r)
    print(f"    pool={len(pool)}  (ambiguous_only={GRPO_AMBIGUOUS_ONLY})")
    for cat, recs in grpo_groups.items():
        print(f"    {cat:25s}: {len(recs):4,}개")
    if not pool:
        print("⚠ 학습 대상 0건 — GRPO_AMBIGUOUS_ONLY 또는 데이터 확인"); return

    # ★ v3: 카테고리 전역 커버 — 평탄화 + 셔플 (max_steps가 첫 카테고리만 소진하는 것 방지)
    flat = list(pool)
    _r.Random(42).shuffle(flat)
    print(f"    → flat={len(flat)} (shuffled, 카테고리 혼합)")

    # ── [3/5] 프롬프트 컨텍스트 파서 + reward ──
    _FIT  = re.compile(r"-\s*(\w+):\s*value=([^,]+),\s*z=([+-]?[\d.]+)\s*\[(ok|DEVIANT)\]")
    _AGD  = re.compile(r"agrees:\s*([^|\n]*)\|\s*differs:\s*([^\n]*)")
    _DIR  = re.compile(r"^(cluster_mismatch|neighbor_insufficient)\s*:")
    _DIRB = re.compile(r"^(cluster_mismatch|neighbor_insufficient)\b")        # colon 불요구
    _DIRC = re.compile(r"^(cluster_mismatch|neighbor_insufficient)\s*:\s*\S") # 완전형
    _ZVAL = re.compile(r"z\s*=\s*[+-]?\d|value\s*=\s*[+-]?\d")   # ★ v4-①: 값 인용 여부

    def _feat_set(s):
        """'source(2/3), ip_country' → {'source','ip_country'}  (카운트 표기 제거)."""
        if not s or not s.strip() or s.strip().startswith("(none)"):
            return set()
        out = set()
        for tok in s.split(","):
            t = re.sub(r"\(.*?\)", "", tok).strip().lower()
            if t:
                out.add(t)
        return out

    def _ctx_from_user(user):
        """user 프롬프트에서 CLUSTER FIT / RETRIEVAL 신호 복원.
        ★ instruction 텍스트에 리터럴 '[DEVIANT]'/'[spread]'가 있으므로
          반드시 해당 '## 블록' 안에서만 파싱한다."""
        u = user or ""
        _ic, _ir = u.find("## CLUSTER"), u.find("## RETRIEVAL")
        fit  = u[_ic:_ir] if (0 <= _ic < _ir) else (u[_ic:] if _ic >= 0 else "")
        retr = u[_ir:] if _ir >= 0 else ""

        dev, ok = set(), set()
        for f, _v, _z, tag in _FIT.findall(fit):          # ★ CLUSTER 블록에서만
            (dev if tag == "DEVIANT" else ok).add(f.lower())

        m = _AGD.search(retr)                             # ★ RETRIEVAL 블록에서만
        agree_set  = _feat_set(m.group(1)) if m else set()
        differ_set = _feat_set(m.group(2)) if m else set()
        return {"deviant": dev, "ok": ok,
                "spread": "[spread]" in retr,             # ★ instruction 오탐 차단
                "agree_set": agree_set, "differ_set": differ_set,
                "agree": len(agree_set), "differ": len(differ_set),
                "has_agd": bool(m), "has_retr": _ir >= 0}

    def _parse_pred(text):
        t = text
        if "```" in t:
            t = t.split("```json")[-1].split("```")[0] if "```json" in t else t.split("```")[1]
        if "{" not in t or "}" not in t:
            return None
        try:
            return json.loads(t[t.index("{"): t.rindex("}") + 1])
        except Exception:
            return None

    def _cited(text, ctx):
        low = (text or "").lower()
        return {f for f in (ctx["deviant"] | ctx["ok"]) if f in low}

    W_AGENT   = {"cls": 0.30, "grounded": 0.30, "direction": 0.20, "format": 0.20}
    W_FORWARD = {"cls": 0.30, "grounded": 0.30, "calib": 0.20, "format": 0.20}
    P_TEMPLATE, P_LENGTH, P_DIST = 0.30, 0.10, 0.20

    def compute_reward_task(text, rec, ctx, seen=None):
        pred = _parse_pred(text)
        if pred is None:
            return 0.0, {"parse": 0.0, "total": 0.0}
        gt = int(rec["metadata"]["gt_label"])

        if GRPO_TASK == "agent":
            # ═══ v6: 사다리(ladder) reward — wm = "검색 품질 평가"로 정렬 ═══
            #  v5 실패 교훈(실측): direction 단독 0.375 안전지대 → P1 0% /
            #  mining-GT r_dir(Δ0.35)가 최대 분산원 → direction 찍기(cm:ni 57:39)만 학습 /
            #  dup 페널티가 direction-flip을 다양성으로 보상(신조어 "neighbor_spread" 포함).
            wm = pred.get("what_missing")
            if not isinstance(wm, str) or not wm.strip():
                return 0.0, {"base": 0.0, "total": 0.0}
            wm = wm.strip()
            m_dir     = _DIRB.match(wm)
            direction = m_dir.group(1) if m_dir else None
            has_col   = bool(_DIRC.match(wm))
            evid      = wm.split(":", 1)[1].strip() if (has_col and ":" in wm) else ""
            cited     = _cited(evid or wm, ctx)

            # ── 사다리 base: 형식이탈 0.05 / direction 단독 0.10(★안전지대 제거) /
            #                evidence 시도(콜론+2단어 이상) 0.30 ──
            if direction is None:
                base = 0.05
            elif not has_col or len(evid.split()) < 2:
                base = 0.10
            else:
                base = 0.30

            has_evi = base >= 0.30            # 이하 모든 가산은 evidence 존재에 조건화

            # ── +0.20: 실제 input feature 인용 (환각 인용은 _cited가 걸러냄) ──
            r_cite = 0.20 * float(has_evi and bool(cited))

            # ── +0.25: 올바른 채널 인용 + 조건 성립 ──
            r_chan = 0.0
            if has_evi and direction == "cluster_mismatch":
                r_chan = 0.25 * float(bool(cited & ctx["deviant"]))
            elif has_evi and direction == "neighbor_insufficient":
                _hit = bool(cited & ctx["differ_set"]) if ctx["differ_set"] else bool(cited)
                _ok  = ctx["spread"] or (ctx["differ"] > 0 and ctx["differ"] >= ctx["agree"])
                r_chan = 0.25 * float(_hit and _ok) + 0.10 * float(_hit and not _ok)
                r_chan = min(r_chan, 0.25)

            # ── +0.15: 구체성 — z/value 명시 ──
            r_val = 0.15 * float(has_evi and bool(_ZVAL.search(evid)))

            # ── +0.10: direction ↔ 결정론 신호 일관성 (GT 아님 — F7 귀결)
            #    +0.05: mining GT 일치는 소액 보너스로 강등 ──
            _sig_cm = len(ctx["deviant"]) > 0
            _sig_ni = ctx["spread"] or (ctx["differ"] > 0 and ctx["differ"] >= ctx["agree"])
            _consis = (direction == "cluster_mismatch" and _sig_cm) or \
                      (direction == "neighbor_insufficient" and _sig_ni)
            r_dir = 0.10 * float(has_evi and _consis)
            md = rec["metadata"]
            if has_evi and md.get("_mining_type", "clean") not in ("clean", None, "?"):
                _gt_dir = "cluster_mismatch" if md.get("_misroute") else "neighbor_insufficient"
                r_dir += 0.05 * float(direction == _gt_dir)

            total = min(base + r_cite + r_chan + r_val + r_dir, 1.0)
            free  = evid or wm
            parts = {"base": base, "cite": r_cite, "chan": r_chan,
                     "val": r_val, "dir": r_dir, "total": total}

        else:                                                   # forward — reasoning 최적화
            cls = str(pred.get("cls", "")).upper()
            if cls not in ("NORMAL", "ABNORMAL"):
                return 0.0, {"parse": 1.0, "format": 0.0, "total": 0.0}
            r_cls = float((cls == "ABNORMAL") == bool(gt))
            rsn   = str(pred.get("reasoning", "")).strip()
            conf  = pred.get("confidence")
            cited, is_ab = _cited(rsn, ctx), (cls == "ABNORMAL")
            # ── 사다리형 grounding: 인용 0.40 + 방향정합 0.35 + z/value 0.25 ──
            _dir_ok = bool(cited) and any((c in ctx["deviant"]) == is_ab for c in cited)
            r_gr = 0.40 * float(bool(cited)) + 0.35 * float(_dir_ok) \
                 + 0.25 * float(_dir_ok and bool(_ZVAL.search(rsn)))
            p = (conf / 100.0) if isinstance(conf, (int, float)) else 0.5
            r_cal = p if r_cls else (1.0 - p)
            r_fmt = 0.5 * float(set(pred) == {"cls", "confidence", "reasoning"}) \
                  + 0.5 * float(bool(rsn))
            # reasoning 비중↑ (grounded 0.30→0.35), calib 0.20→0.15
            total = 0.30 * r_cls + 0.35 * r_gr + 0.15 * r_cal + 0.20 * r_fmt
            free  = rsn
            parts = {"cls": r_cls, "grounded": r_gr, "calib": r_cal,
                     "format": r_fmt, "total": total}

        # ── 공통 페널티 ──
        low = (free or "").lower()
        if "distance" in low or "boundary" in low:              # 거리 재진술 금지 (v2 규칙)
            total -= P_DIST
        # ★ dup 페널티는 evidence/reasoning 있는 출력에만 — direction-flip 보상 차단
        if (GRPO_TASK == "forward") or (parts.get("base", 0) >= 0.30):
            key = re.sub(r"[^a-z0-9 ]", "", low)[:40]
            if seen is not None and key:
                seen[key] = seen.get(key, 0) + 1
                if seen[key] > 1:
                    total -= P_TEMPLATE
        if len(student_tokenizer(text).input_ids) > GRPO_LEN_BUDGET:
            total -= P_LENGTH
        parts["total"] = max(total, 0.0)
        return parts["total"], parts

    # ── [4/5] 생성 / log-prob ──
    def generate_group(prompts):
        """B개 프롬프트 → B×G 시퀀스. attention mask까지 정확히 구성."""
        orig_side = student_tokenizer.padding_side
        student_tokenizer.padding_side = "left"
        full = [f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n" for p in prompts]
        inp = student_tokenizer(full, return_tensors="pt", padding=True,
                                truncation=True, max_length=1536).to(student_model.device)
        plen = inp.input_ids.shape[1]
        with torch.no_grad():
            seqs = student_model.generate(
                **inp, max_new_tokens=GRPO_MAX_NEW, do_sample=True,
                temperature=GRPO_TEMPERATURE, num_return_sequences=GRPO_NUM_GENERATIONS,
                use_cache=True, pad_token_id=student_tokenizer.pad_token_id,
                eos_token_id=student_tokenizer.eos_token_id)
        gen_len = seqs.shape[1] - plen
        attn = torch.cat([inp.attention_mask.repeat_interleave(GRPO_NUM_GENERATIONS, 0),
                          torch.ones(seqs.shape[0], gen_len, dtype=inp.attention_mask.dtype,
                                     device=seqs.device)], dim=1)
        comps = [student_tokenizer.decode(s[plen:], skip_special_tokens=True)
                 .split("<|im_end|>")[0].strip() for s in seqs]
        student_tokenizer.padding_side = orig_side
        return seqs, attn, plen, comps

    def _seq_log_probs(model, seqs, attn, plen, grad):
        """생성 토큰 구간의 평균 log-prob (시퀀스별)."""
        out = []
        for i in range(seqs.shape[0]):
            seq, am = seqs[i].unsqueeze(0), attn[i].unsqueeze(0)
            if grad:
                o = model(input_ids=seq, attention_mask=am, use_cache=False)
            else:
                with torch.no_grad():
                    o = model(input_ids=seq, attention_mask=am, use_cache=False)
            lp  = F.log_softmax(o.logits[0, plen - 1:-1, :], dim=-1)
            gen = seqs[i][plen:][:lp.shape[0]]
            if len(gen) == 0:
                out.append(torch.zeros((), device=seqs.device)); continue
            out.append(lp[torch.arange(len(gen), device=seqs.device), gen].mean())
        return out

    print("\n[3/5] GRPO Configuration")
    print(f"  reward   : cls/grounded/{'direction' if GRPO_TASK=='agent' else 'calib'}/format "
          f"− template·length·distance   (v4: z/value 인용 필수)")
    print(f"  G={GRPO_NUM_GENERATIONS}  B={GRPO_BATCH_SIZE}  epochs={GRPO_EPOCHS}  "
          f"max_steps={GRPO_MAX_STEPS}  max_new={GRPO_MAX_NEW}  len_budget={GRPO_LEN_BUDGET}")

    optimizer = AdamW([p for p in student_model.parameters() if p.requires_grad], lr=GRPO_LR)

    print("\n[4/5] Starting GRPO training...")

    # ── [5/5] 학습 루프 (평탄화된 flat 단일 루프) ──
    for epoch in range(GRPO_EPOCHS):
        print(f"\n{'='*60}\nEpoch {epoch+1}/{GRPO_EPOCHS}\n{'='*60}")
        student_model.train()
        epoch_rewards, epoch_losses = [], []
        step, n_skip, n_upd = 0, 0, 0

        _total = min(len(flat), (GRPO_MAX_STEPS or 10**9) * GRPO_BATCH_SIZE)
        for bs in tqdm(range(0, _total, GRPO_BATCH_SIZE), desc=f"  epoch{epoch+1}"):
            if GRPO_MAX_STEPS and step >= GRPO_MAX_STEPS:
                break
            batch = flat[bs:bs + GRPO_BATCH_SIZE]
            def _u_of(r):
                """SFT(to_chatml)와 동일 입력 정합: agent는 sufficiency 미주입 시 보충 (이중 주입 방지)."""
                u = r["user"]
                if GRPO_TASK == "agent" and "retrieval_sufficiency =" not in u:
                    _nd = r["metadata"].get("top1_l2_dist_norm", -1)
                    _suf = "ambiguous" if 0.3 <= _nd < 0.55 else "confident"
                    u = u + f"\n\n[SYSTEM-COMPUTED] retrieval_sufficiency = {_suf.upper()}"
                return u
            prompts = [_u_of(r) for r in batch]         # ★ metadata['user_prompt'] 아님

            t0 = time.time()
            seqs, attn, plen, comps = generate_group(prompts)
            t_gen = time.time() - t0

            t0 = time.time()
            pol_lp = torch.stack(_seq_log_probs(student_model, seqs, attn, plen, grad=True))
            with student_model.disable_adapter():        # ★ 같은 시퀀스에 대한 ref log-prob
                ref_lp = torch.stack(
                    _seq_log_probs(student_model, seqs, attn, plen, grad=False)).detach()
            t_ref = time.time() - t0

            t0 = time.time()
            rewards, parts0 = [], None
            for i, rec in enumerate(batch):
                ctx, seen = _ctx_from_user(rec["user"]), {}   # 중복 페널티는 그룹 내에서
                for g in range(GRPO_NUM_GENERATIONS):
                    r, pt = compute_reward_task(comps[i * GRPO_NUM_GENERATIONS + g],
                                                rec, ctx, seen)
                    rewards.append(r)
                    if parts0 is None:
                        parts0 = pt
            t_rew = time.time() - t0

            R = torch.tensor(rewards, dtype=torch.float32, device=student_model.device) \
                     .view(len(batch), GRPO_NUM_GENERATIONS)
            adv = ((R - R.mean(1, keepdim=True))
                   / (R.std(1, keepdim=True) + 1e-8)).view(-1)      # ★ group-relative

            # ── 첫 스텝 진단 (skip 여부와 무관하게 항상 출력) ──
            if step == 0:
                print(f"\n⏱️ Gen:{t_gen:.1f}s Ref:{t_ref:.1f}s Rew:{t_rew:.1f}s "
                      f"| Total:{t_gen+t_ref+t_rew:.1f}s")
                print(f"   rewards={['%.2f' % x for x in rewards]}  adv_std={adv.std().item():.3f}")
                print(f"   reward parts[0]: { {k: round(v,2) for k, v in (parts0 or {}).items()} }")
                print(f"   category: {batch[0]['metadata'].get('category')}")
                print(f"   completion[0]: {comps[0][:160]!r}")
                _c = _ctx_from_user(batch[0]["user"])
                print(f"   ctx: deviant={_c['deviant']} ok={_c['ok']} spread={_c['spread']} "
                      f"agree={_c['agree_set'] or '∅'} differ={_c['differ_set'] or '∅'}")
                # ★ v4-③: RETRIEVAL 블록 원문 진단 (agrees/differs 라인 존재 확인)
                _tail = batch[0]["user"]
                _i = _tail.find("## RETRIEVAL")
                print(f"   RETRIEVAL 원문: {(_tail[_i:_i+220] if _i >= 0 else _tail[-220:])!r}")
                if not _c["has_agd"]:
                    print("   ⚠ agrees/differs 라인 없음 → neighbor_insufficient 근거 채널 부재. "
                          "prompts.format_retrieval_digest의 topk_feats 전달 확인")
                if not _c["deviant"] and not _c["ok"]:
                    print("   ⚠ FIT 파싱 실패(deviant/ok 공집합) → _FIT 정규식 vs 프롬프트 포맷 확인")

            # ── zero-variance skip: adv=0이면 KL만 남아 base로 되돌림 → 업데이트 생략 ──
            if adv.abs().max().item() < 1e-6:
                n_skip += 1
                step += 1
                epoch_rewards.extend(rewards)          # 지표에는 반영
                continue

            t0 = time.time()
            ratio = torch.exp(pol_lp - ref_lp)
            kl    = GRPO_KL_COEF * (ratio - (pol_lp - ref_lp) - 1).mean()
            loss  = -(adv * pol_lp).mean() + kl

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
            optimizer.step()
            t_opt = time.time() - t0

            epoch_rewards.extend(rewards)
            epoch_losses.append(loss.item())
            step += 1
            n_upd += 1

        if epoch_rewards:
            _loss = (sum(epoch_losses) / len(epoch_losses)) if epoch_losses else float("nan")
            print(f"\n📊 Epoch {epoch+1}: Reward={sum(epoch_rewards)/len(epoch_rewards):.3f} "
                  f"Loss={_loss:.3f}  steps={step}  updated={n_upd}  skipped(adv=0)={n_skip}")
            if n_upd == 0:
                print("   ⚠ 업데이트 0회 — 모든 그룹의 reward가 동일. "
                      "reward가 바닥/천장에 고착했는지(진단 출력) 또는 G↑/temperature↑ 검토")

    print("\n[5/5] GRPO Complete!")

    grpo_path = f"{LOCAL_ADAPTER}_grpo"
    student_model.save_pretrained(grpo_path)
    student_tokenizer.save_pretrained(grpo_path)
    print(f"✅ Saved: {grpo_path}")

In [29]:
# ============================================================
# Section 11: 학습 실행 — SFT vs GRPO 분기
# ============================================================

if not USE_GRPO:
    # ──────────────────────────────────────────────────────
    # SFT 학습 (Curriculum ON/OFF 지원)
    # ──────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("Training Mode: SFT")
    print("="*60)

    if USE_CURRICULUM:
        # ── Curriculum Learning: 2-Stage ─────────────────
        print("\n[Stage 1] Easy Samples Only")

        # Easy samples 추출
        easy_categories = {"easy_normal", "easy_abnormal"}
        easy_records = [r for r in training_records
                       if r['metadata']['category'] in easy_categories]
        easy_dataset = [to_chatml(r) for r in easy_records]

        trainer.train_dataset = easy_dataset
        trainer.args.num_train_epochs = CURRICULUM_STAGE1_EPOCHS
        trainer.train()

        print("\n[Stage 2] Easy + Hard Samples")

        # Full dataset
        full_dataset = [to_chatml(r) for r in training_records]
        trainer.train_dataset = full_dataset
        trainer.args.num_train_epochs = CURRICULUM_STAGE2_EPOCHS

        # Stage 1 checkpoint에서 재시작
        last_ckpt = max(Path(LOCAL_ADAPTER).glob("checkpoint-*"),
                       key=lambda p: int(p.name.split("-")[-1]))
        print(f"  Resuming from: {last_ckpt}")
        trainer.train(resume_from_checkpoint=str(last_ckpt))

    else:
        # ── Single Stage: All Samples ────────────────────
        print("\n[Single Stage] All Samples")
        trainer.train()

    print("\n✅ SFT Training Complete!")

else:
    if GRPO_FROM_SAVED_SFT:
        # ── 저장된 SFT adapter 로드 → GRPO 시작점 (매번 SFT 선행하는 비효율 제거) ──
        import gc, glob, shutil, torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel

        _SFT_GCS = f"{os.path.dirname(OUT_ADAPTER_PATH.rstrip('/'))}/{model_tag}adapter_sft"
        _src = (_SFT_GCS if os.path.exists(os.path.join(_SFT_GCS, "adapter_config.json"))
                else (sorted(glob.glob(os.path.join(_SFT_GCS, "checkpoint-*")))[-1]
                      if glob.glob(os.path.join(_SFT_GCS, "checkpoint-*")) else None))
        assert _src, f"SFT adapter 없음: {_SFT_GCS} — 먼저 USE_GRPO=False로 SFT 1회 실행 필요"
        _LOCAL = "/content/sft_adapter_for_grpo"
        shutil.rmtree(_LOCAL, ignore_errors=True); shutil.copytree(_src, _LOCAL)

        # ★ 이중 래핑 방지: trainer가 래핑한 모델을 버리고 새 base에 adapter 장착
        for _v in ["trainer", "student_model"]:
            if _v in dir(): exec(f"del {_v}")
        gc.collect(); torch.cuda.empty_cache()

        if "student_tokenizer" not in dir():               # 블록 2 생략 시 대비
            student_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
            student_tokenizer.pad_token = student_tokenizer.eos_token
            student_tokenizer.truncation_side = "left"
        _base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
        student_model = PeftModel.from_pretrained(_base, _LOCAL, is_trainable=True)  # ★ 학습 가능
        student_model.train()
        print(f"✅ SFT adapter 로드(GRPO 시작점): {_src}")

    print(f"Training Mode: GRPO [{GRPO_TASK}]")
    run_grpo_training()
    # print(f"Training Mode: GRPO [{GRPO_TASK}]")
    # run_grpo_training()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Training Mode: SFT

[Single Stage] All Samples


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.093552,0.097036,0.106300,512502.000000,0.961523
2,0.091800,0.090990,0.096464,1025004.000000,0.963419


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.093552,0.097036,0.106300,512502.000000,0.961523
2,0.091800,0.090990,0.096464,1025004.000000,0.963419
3,0.086796,0.089479,0.096415,1537506.000000,0.963656



✅ SFT Training Complete!


In [30]:
# cell 66 (학습) 직후, 같은 student_model로 즉시 1개 생성 테스트
student_model.eval()
r = training_records[0]
prompt = student_tokenizer.apply_chat_template(
    [{"role":"system","content":r['system']},{"role":"user","content":r['user']}],
    tokenize=False, add_generation_prompt=True)
inputs = student_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(student_model.device)
out = student_model.generate(**inputs, max_new_tokens=GEN_MAX_NEW, do_sample=False,
                             pad_token_id=student_tokenizer.pad_token_id)
print(student_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
print("---expected (train label):", r['assistant'])

{"cls": "NORMAL", "sufficiency": "confident", "what_missing": null}
---expected (train label): {"cls": "NORMAL", "sufficiency": "confident", "what_missing": null}


## Section 12: adapter GCS 업로드

In [31]:
# ============================================================
# Section 12: 모델 GCS 업로드 (SFT/GRPO 통합)
# ============================================================

import shutil

# ──────────────────────────────────────────────────────
# 1. 업로드할 모델 경로 결정
# ──────────────────────────────────────────────────────
if USE_GRPO:
    # GRPO 모델 (최종 저장본)
    source_model_path = f"{LOCAL_ADAPTER}_grpo"
    method_suffix = "grpo"
    best_metric = None  # GRPO는 best checkpoint 개념 없음

    print(f"\n📦 GRPO 모델 업로드 준비")
    print(f"   Source: {source_model_path}")

else:
    # SFT 모델 (best checkpoint)
    method_suffix = "sft"

    best_ckpt = trainer.state.best_model_checkpoint
    if best_ckpt is None:
        import glob
        ckpt_dirs = sorted(glob.glob(os.path.join(LOCAL_ADAPTER, "checkpoint-*")))
        if ckpt_dirs:
            best_loss, best_ckpt = float("inf"), ckpt_dirs[-1]
            for ckpt in ckpt_dirs:
                state_path = os.path.join(ckpt, "trainer_state.json")
                if os.path.exists(state_path):
                    with open(state_path) as f:
                        s = json.load(f)
                    loss = s.get("best_metric") or float("inf")
                    if loss < best_loss:
                        best_loss, best_ckpt = loss, ckpt
        else:
            best_ckpt = LOCAL_ADAPTER

    source_model_path = best_ckpt
    best_metric = trainer.state.best_metric

    print(f"\n📦 SFT 모델 업로드 준비")
    print(f"   Best checkpoint: {source_model_path}")
    print(f"   Best eval loss:  {best_metric}")

# ──────────────────────────────────────────────────────
# 2. GCS 경로 설정 (메서드별 폴더 구분)
# ──────────────────────────────────────────────────────
# 예: /content/gcs_mount/.../hf_0418/sft/ 또는 /grpo/
# gcs_method_path = OUT_ADAPTER_PATH.rstrip('/') + f"_{method_suffix}"
gcs_method_path = f"{os.path.dirname(OUT_ADAPTER_PATH.rstrip('/'))}/{model_tag}adapter_{method_suffix}"

print(f"   Destination: {gcs_method_path}")

# ──────────────────────────────────────────────────────
# 3. 재귀 복사 함수 (기존 재사용)
# ──────────────────────────────────────────────────────
def copy_recursive(src_dir: str, dst_dir: str) -> float:
    """재귀적으로 디렉토리 복사, 총 크기(KB) 반환"""
    os.makedirs(dst_dir, exist_ok=True)
    total_kb = 0.0
    for entry in os.listdir(src_dir):
        src = os.path.join(src_dir, entry)
        dst = os.path.join(dst_dir, entry)
        if os.path.isdir(src):
            print(f"  📁 {entry}/")
            total_kb += copy_recursive(src, dst)
        else:
            shutil.copy2(src, dst)
            size_kb = os.path.getsize(dst) / 1024
            total_kb += size_kb
            print(f"    ↑ {entry} ({size_kb:.1f} KB)")
    return total_kb

# ──────────────────────────────────────────────────────
# 4. 업로드 실행
# ──────────────────────────────────────────────────────
TOKENIZER_FILES = {
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt"
}

print(f"\n업로드 시작: {source_model_path} → {gcs_method_path}")
total_kb = copy_recursive(source_model_path, gcs_method_path)

# Tokenizer 파일 추가 복사 (root에 있을 수 있음)
for fname in TOKENIZER_FILES:
    # SFT: LOCAL_ADAPTER에 있음
    # GRPO: source_model_path에 이미 복사됨
    src = os.path.join(LOCAL_ADAPTER if not USE_GRPO else source_model_path, fname)
    dst = os.path.join(gcs_method_path, fname)

    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(dst) / 1024
        total_kb += size_kb
        print(f"  ↑ {fname} (tokenizer, {size_kb:.1f} KB)")

print(f"\n✅ 업로드 완료: {gcs_method_path}")
print(f"   총 크기: {total_kb/1024:.1f} MB")
print(f"   메서드: {method_suffix.upper()}")

# ──────────────────────────────────────────────────────
# 5. (선택) 메타데이터 저장
# ──────────────────────────────────────────────────────
from datetime import datetime
metadata = {
    "method": method_suffix,
    "use_curriculum": USE_CURRICULUM,
    "use_mining": USE_MINING,
    "base_model": BASE_MODEL_ID,
    "experiment_case": EXPERIMENT_CASE,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

if USE_GRPO:
    metadata.update({
        "grpo_epochs": GRPO_EPOCHS,
        "grpo_lr": GRPO_LR,
        "grpo_batch_size": GRPO_BATCH_SIZE,
        "grpo_num_generations": GRPO_NUM_GENERATIONS,
    })
else:
    metadata.update({
        "sft_epochs": sft_config.num_train_epochs,
        "best_eval_loss": float(best_metric) if best_metric else None,
    })

metadata_path = os.path.join(gcs_method_path, "training_metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n📄 메타데이터 저장: training_metadata.json")


📦 SFT 모델 업로드 준비
   Best checkpoint: /content/adapter-pca_64_k100/checkpoint-375
   Best eval loss:  0.0894787386059761
   Destination: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/agent_s3_adapter_sft

업로드 시작: /content/adapter-pca_64_k100/checkpoint-375 → /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/agent_s3_adapter_sft
    ↑ adapter_model.safetensors (8472.7 KB)
    ↑ scheduler.pt (1.4 KB)
    ↑ tokenizer_config.json (0.7 KB)
    ↑ training_args.bin (5.6 KB)
    ↑ optimizer.pt (17053.8 KB)
    ↑ adapter_config.json (1.0 KB)
    ↑ rng_state.pth (14.3 KB)
    ↑ README.md (5.1 KB)
    ↑ chat_template.jinja (2.4 KB)
    ↑ trainer_state.json (23.9 KB)
    ↑ tokenizer.json (11154.2 KB)

✅ 업로드 완료: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/agent_s3_adapter_sft
   총 크기: 35.9 MB
   메서드: SFT

📄 메타데이터 저장: training_metadata.json


In [ ]:
print(f"{'='*60}")
print(f"Fine-tuning 완료 요약")
print(f"{'='*60}")
print(f"  experiment_case  : {EXPERIMENT_CASE}")
print(f"  base_model       : {BASE_MODEL_ID}")
print(f"  teacher_provider : {TEACHER_PROVIDER.upper()}")
print(f"  teacher_model    : {TEACHER_MODEL}")
# print(f"  sampling_method  : {'pipeline_mined' if USE_MINED_NEGATIVES else 'distance_based'}")
# print(f"  mined_parquet    : {MINED_PARQUET_PATH if USE_MINED_NEGATIVES else 'N/A'}")
# print(f"  run_tag          : {RUN_TAG}")
print(f"  학습 데이터      : {len(training_records)}개")
print(f"    NORMAL         : {cls_list.count('NORMAL')}")
print(f"    ABNORMAL       : {cls_list.count('ABNORMAL')}")
# print(f"  training_data GCS: {OUT_JSONL_PATH}")
print(f"  adapter GCS      : {gcs_method_path}")
print(f"{'='*60}")
print("\n[다음 단계]")
print("  1. Qwen2.5-0.5B base + adapter merge → GGUF Q4_K_M 변환")
print("  2. llama-cpp-python으로 GKE CPU 서빙")
print("  3. analyzer.py Gemini 호출 → 로컬 모델 호출로 교체")


In [ ]:
# ── 평가 완료 후 Teacher GPU 해제 ──────────────────────────────────
if TEACHER_PROVIDER == "huggingface":
    release_teacher_gpu()
    print("✅ 평가 완료 후 Teacher GPU 해제")

## Section 13: GGUF

In [34]:
# ============================================================
# Section 14: GGUF 변환 + 업로드 (CPU 서빙용)
#   SFT/GRPO로 학습된 student_model을 병합 → GGUF Q4_K_M → GCS
#   저장 위치: {gcs_method_path}/gguf/  (adapter와 같은 경로 체계)
# ============================================================
import os, gc, glob, shutil

# [1] 패키지 (--no-deps로 torch/numpy 보호)
!pip install -q llama-cpp-python --no-deps
!pip install -q diskcache gguf

# [2] llama.cpp 빌드
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp
!apt-get install -y -q cmake
!cd llama.cpp && mkdir -p build && cd build && cmake .. -DLLAMA_CURL=OFF && make -j

# [3] LoRA 병합 → fp16 (student_model = 학습된 PeftModel)
GGUF_WORK   = "./gguf_work"
merged_path = f"{GGUF_WORK}/merged_fp16"
os.makedirs(merged_path, exist_ok=True)

print("🔄 LoRA 병합 중...")
# GRPO면 grpo adapter를, 아니면 SFT adapter를 base에 병합
# _adapter_src = (f"{LOCAL_ADAPTER}_grpo" if USE_GRPO else LOCAL_ADAPTER)
import glob, os
# adapter_config.json이 있는 실제 경로 찾기 (LOCAL_ADAPTER 직하 or checkpoint-NN)
if USE_GRPO:
    _adapter_src = f"{LOCAL_ADAPTER}_grpo"
elif os.path.exists(os.path.join(LOCAL_ADAPTER, "adapter_config.json")):
    _adapter_src = LOCAL_ADAPTER
else:
    # checkpoint-NN 중 최신 (step 번호 max)
    _ckpts = glob.glob(os.path.join(LOCAL_ADAPTER, "checkpoint-*"))
    _adapter_src = max(_ckpts, key=lambda p: int(p.split("-")[-1]))
print(f"adapter 경로: {_adapter_src}")
assert os.path.exists(os.path.join(_adapter_src, "adapter_config.json")), \
    f"adapter_config.json 없음: {_adapter_src}"

from peft import PeftModel
_base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
_merge = PeftModel.from_pretrained(_base_fp16, _adapter_src)
_merge = _merge.merge_and_unload()
_merge.save_pretrained(merged_path)
student_tokenizer.save_pretrained(merged_path)
del _merge, _base_fp16; gc.collect(); torch.cuda.empty_cache()
print(f"✅ 병합 fp16 저장: {merged_path}")

# [4] HF → GGUF fp16 → Q4_K_M 양자화
gguf_fp16 = f"{GGUF_WORK}/model_fp16.gguf"
gguf_q4   = f"{GGUF_WORK}/model_q4.gguf"
QUANTIZE_BIN = "./llama.cpp/build/bin/llama-quantize"

!python3 llama.cpp/convert_hf_to_gguf.py {merged_path} --outfile {gguf_fp16} --outtype f16
!{QUANTIZE_BIN} {gguf_fp16} {gguf_q4} Q4_K_M
print(f"✅ GGUF Q4_K_M 생성: {gguf_q4}")

# [5] GCS 업로드 — adapter와 같은 경로 체계 아래 gguf/
#     gcs_method_path = OUT_ADAPTER_PATH + "_{sft|grpo}" (cell 25에서 정의됨)
gguf_gcs_dir = f"{gcs_method_path.rstrip('/')}/gguf"
gguf_gcs     = f"{gguf_gcs_dir}/model_q4.gguf"

print(f"🔄 GCS 업로드: {gguf_gcs}")
# 기존 cell 25가 gsutil/blob 어느 방식이든 — 동일 방식 사용
!gsutil cp {gguf_q4} {gguf_gcs}
# 양자화 레벨 기록
import json as _json
_meta = {"base_model": BASE_MODEL_ID, "quant": "Q4_K_M",
         "method": ("grpo" if USE_GRPO else "sft"),
         "size_mb": round(os.path.getsize(gguf_q4)/1024**2, 1)}
with open(f"{GGUF_WORK}/gguf_meta.json","w") as f: _json.dump(_meta, f)
!gsutil cp {GGUF_WORK}/gguf_meta.json {gguf_gcs_dir}/gguf_meta.json
print(f"✅ GGUF 업로드 완료: {gguf_gcs}  ({_meta['size_mb']} MB)")

Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 161 not upgraded.
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.17.0
-- ggml commit:  e8e6c7af2
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.4s)
-- Generating done (0.3s)
-- Build files have been written to: /content/llama.cpp/build
[  1%] Built target sha256
[  2%] Built target llama-common-base
[  2%] Built target xxhash
[  2%] Built target sha1
[  2%] Built target llama-ui-embed
[  2%] Built target llama-llava-cli
[  2%] Built target llama-qwen2vl-cli
[  2%] Built t

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 병합 fp16 저장: ./gguf_work/merged_fp16
INFO:hf-to-gguf:Loading model: merged_fp16
INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> 

In [35]:
# ============================================================
# Section 13: 학습 직후 인라인 검증 (NB03 없이 빠른 확인)
#   목적: TRAIN_MODE(cls / cls_suf)가 collapse 없이 학습됐나
#   - student_model 메모리에 살아있을 때 바로 추론 (GGUF 변환 불필요)
#   - cls는 nd기준/GT기준 분리, cls_suf면 sufficiency도
# ============================================================
import json as _json
from collections import Counter

N_EVAL_SAMPLES = min(50, len(eval_recs))
NORM_CLEAR_LOW, NORM_CLEAR_HIGH = 0.3, 0.55   # boundary 경계 (prompts.py/local_analyzer 동일 기준)

student_model.eval()

def _split_chatml(text: str):
    prompt   = text.split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    expected = text.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0]
    return prompt, expected

def _try_parse(txt: str):
    try:
        t = txt.split("```json")[-1].split("```")[0].strip() if "```" in txt else txt.strip()
        return _json.loads(t)
    except Exception:
        return None

rows = []
for r in eval_recs[:N_EVAL_SAMPLES]:
    chatml_text = to_chatml(r)["text"]          # WM_FOCUS 반영된 실제 학습 포맷 그대로
    prompt, expected_str = _split_chatml(chatml_text)
    expected = _try_parse(expected_str) or {}

    inputs = student_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(student_model.device)
    with torch.no_grad():
        out = student_model.generate(
            **inputs, max_new_tokens=GEN_MAX_NEW, do_sample=False,
            pad_token_id=student_tokenizer.pad_token_id,
        )
    pred_txt = student_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = _try_parse(pred_txt)

    nd = r["metadata"].get("top1_l2_dist_norm", -1)
    nd_label = None
    if nd >= 0:
        if nd < NORM_CLEAR_LOW:   nd_label = "NORMAL"
        elif nd >= NORM_CLEAR_HIGH: nd_label = "ABNORMAL"
        # boundary 구간은 nd_label=None (nd 자체가 무정보 구간)

    rows.append({
        "gt":        "ABNORMAL" if r["metadata"]["gt_label"] == 1 else "NORMAL",
        "nd":        nd,
        "nd_label":  nd_label,
        "zone":      "boundary" if nd_label is None and nd >= 0 else "nd_clear",
        "pred":      pred,
        "parse_ok":  pred is not None,
        "pred_cls":  (pred or {}).get("cls"),
        "pred_suf":  (pred or {}).get("sufficiency"),
        "exp_suf":   expected.get("sufficiency"),
        "pred_wm":   (pred or {}).get("what_missing"),
    })

# ── 집계 ────────────────────────────────────────────────────
n_total     = len(rows)
n_parse_ok  = sum(r["parse_ok"] for r in rows)
print(f"[Parse] {n_parse_ok}/{n_total} 성공 ({n_parse_ok/n_total*100:.1f}%)")

# ── collapse 감지: cls 예측이 한쪽으로 쏠려있나 ───────────────
cls_counts = Counter(r["pred_cls"] for r in rows if r["parse_ok"])
print(f"\n[Collapse check] cls 예측 분포: {dict(cls_counts)}")
if cls_counts:
    top_ratio = max(cls_counts.values()) / sum(cls_counts.values())
    print(f"  최빈값 비율: {top_ratio*100:.1f}%  {'⚠️ collapse 의심' if top_ratio > 0.9 else '✅ 정상 분산'}")

# ── nd_clear 구간: pred vs nd(모델이 nd만 베꼈나) / pred vs GT ──
clear = [r for r in rows if r["zone"] == "nd_clear" and r["parse_ok"]]
if clear:
    nd_agree = sum(r["pred_cls"] == r["nd_label"] for r in clear) / len(clear)
    gt_acc_clear = sum(r["pred_cls"] == r["gt"] for r in clear) / len(clear)
    print(f"\n[nd_clear zone] n={len(clear)}")
    print(f"  pred vs nd-derived label 일치율: {nd_agree*100:.1f}%  (모델이 nd 신호를 잘 따라가는지)")
    print(f"  pred vs GT 정확도            : {gt_acc_clear*100:.1f}%")

# ── boundary 구간: GT 정확도만 의미 있음 (nd 자체가 무정보) ────
boundary = [r for r in rows if r["zone"] == "boundary" and r["parse_ok"]]
if boundary:
    gt_acc_boundary = sum(r["pred_cls"] == r["gt"] for r in boundary) / len(boundary)
    print(f"\n[boundary zone] n={len(boundary)}")
    print(f"  pred vs GT 정확도: {gt_acc_boundary*100:.1f}%  "
          f"(teacher 상한선 59.3%와 비교 — 이 근처면 정상, 훨씬 낮으면 collapse)")

# ── sufficiency: WM_FOCUS=False이고 TRAIN_MODE에 sufficiency가 있을 때만 ──
if not WM_FOCUS and TRAIN_MODE in ("cls_suf", "full"):
    suf_rows = [r for r in rows if r["parse_ok"] and r["exp_suf"] is not None]
    if suf_rows:
        suf_acc = sum(r["pred_suf"] == r["exp_suf"] for r in suf_rows) / len(suf_rows)
        suf_counts = Counter(r["pred_suf"] for r in suf_rows)
        print(f"\n[sufficiency] n={len(suf_rows)}  정확도={suf_acc*100:.1f}%  분포={dict(suf_counts)}")
else:
    print(f"\n[sufficiency] WM_FOCUS={WM_FOCUS} → sufficiency는 입력에 deterministic 주입, 모델 예측 대상 아님 (스킵)")

# ── what_missing 다양성 (보일러플레이트 오염 체크, doc1의 42% 이슈) ──
wm_texts = [r["pred_wm"] for r in rows if r["parse_ok"] and r["pred_wm"]]
if wm_texts:
    uniq_ratio = len(set(wm_texts)) / len(wm_texts)
    print(f"\n[what_missing 다양성] unique/total = {len(set(wm_texts))}/{len(wm_texts)} ({uniq_ratio*100:.1f}%)")
    print(f"  최빈 문구: {Counter(wm_texts).most_common(3)}")

[Parse] 50/50 성공 (100.0%)

[Collapse check] cls 예측 분포: {'ABNORMAL': 42, 'NORMAL': 8}
  최빈값 비율: 84.0%  ✅ 정상 분산

[nd_clear zone] n=19
  pred vs nd-derived label 일치율: 94.7%  (모델이 nd 신호를 잘 따라가는지)
  pred vs GT 정확도            : 89.5%

[boundary zone] n=31
  pred vs GT 정확도: 54.8%  (teacher 상한선 59.3%와 비교 — 이 근처면 정상, 훨씬 낮으면 collapse)

[sufficiency] WM_FOCUS=True → sufficiency는 입력에 deterministic 주입, 모델 예측 대상 아님 (스킵)

[what_missing 다양성] unique/total = 25/31 (80.6%)
  최빈 문구: [('neighbor_insufficient: weekday_purchase, month_purchase differ from neighbors; neighbors dense', 4), ('neighbor_insufficient: IP_country, month_purchase differ from neighbors; neighbors dense', 3), ('neighbor_insufficient: sex, month_purchase differ from neighbors; neighbors dense', 2)]
